In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import sklearn

from mhealth_activity.recording import Recording
from mhealth_activity.types import Activity
from mhealth_activity import WatchLocation
from mhealth_activity import Trace

# Data exploration #

1. Find the step count ground truths

In [2]:
train_dir = Path("data/train")

gt_files = []

for path in sorted(train_dir.glob("*.pkl")):
    rec = Recording(str(path))
    if rec.labels is None:
        continue
    sc = rec.labels.get("step_count", None)
    if sc is not None and sc != -1:
        gt_files.append(path.name)

# save
with open("stepcount_files.json", "w") as f:
    json.dump(gt_files, f)

print(f"Saved {len(gt_files)} GT files")

Saved 33 GT files


In [3]:
rows = []

for name in gt_files:
    rec = Recording(str(train_dir / name))
    sc = int(rec.labels["step_count"])

    activity_ids = rec.labels.get("activities", [])
    activity_names = [
        activity.name.lower()
        for activity in Activity
        if activity.value in activity_ids
    ]

    phone_steps_trace = rec.data.get("phone_steps", None)
    phone_steps_val = (
        int(phone_steps_trace.values[-1])
        if phone_steps_trace is not None and len(phone_steps_trace.values) > 0
        else None
    )

    rows.append({
        "file": name,
        "gt_steps": sc,
        "phone_steps": phone_steps_val,
        "diff": None if phone_steps_val is None else phone_steps_val - sc,
        "activities": activity_names,
    })

df = pd.DataFrame(rows)
df

,file,gt_steps,phone_steps,diff,activities
0,train_trace_004.pkl,1023,971.0,-52.0,[walking]
1,train_trace_015.pkl,734,NaN,NaN,[]
2,train_trace_017.pkl,0,NaN,NaN,[cycling]
3,train_trace_021.pkl,0,NaN,NaN,[]
4,train_trace_054.pkl,1000,NaN,NaN,[]
5,train_trace_074.pkl,698,NaN,NaN,[]
6,train_trace_086.pkl,0,NaN,NaN,[cycling]
7,train_trace_090.pkl,155,97.0,-58.0,"[walking, cycling]"
8,train_trace_093.pkl,172,340.0,168.0,"[walking, cycling]"
9,train_trace_110.pkl,192,NaN,NaN,[]


# Windowed & peak-based baseline #

Estimate steps from accelerometer magnitude using local peak detection.

Method (brief):

- Compute a_mag, remove mean, bandpass (≈0.7–3 Hz)
- Split into short windows (~10 s)
- Detect peaks per window (distance + prominence)
- Keep windows with plausible cadence (≈1.3–2.8 Hz)
- Sum peaks → step count

In [4]:
from scipy.signal import butter, filtfilt, find_peaks
import numpy as np

# DON'T TOUCH the hyperparameters, they are tuned and won't get better.
def estimate_steps_windowed(
    rec,
    window_s=10.0,
    low_hz=0.7,
    high_hz=3.0,
    peak_prominence=0.14,
    max_step_hz=3.0,
    min_peak_rate_hz=1.3,
    min_std_threshold=0.08,
    min_final_steps_to_keep=80,
):
    ax = rec.data["ax"].values.astype(float)
    ay = rec.data["ay"].values.astype(float)
    az = rec.data["az"].values.astype(float)
    fs = float(rec.data["ax"].samplerate)

    mag = np.sqrt(ax**2 + ay**2 + az**2)
    mag_centered = mag - np.mean(mag)

    nyq = fs / 2.0
    b, a = butter(3, [low_hz / nyq, high_hz / nyq], btype="band")
    filt = filtfilt(b, a, mag_centered)

    win_len = int(window_s * fs)
    min_distance = int(fs / max_step_hz)

    total_steps = 0
    kept_windows = []

    for start in range(0, len(filt), win_len):
        end = min(start + win_len, len(filt))
        segment = filt[start:end]

        if len(segment) < max(10, min_distance):
            continue

        # energy-based gating
        seg_std = float(np.std(segment))
        if seg_std < min_std_threshold:
            continue

        peaks, props = find_peaks(
            segment,
            distance=max(1, min_distance),
            prominence=peak_prominence,
        )

        duration = len(segment) / fs
        peak_rate_hz = len(peaks) / max(duration, 1e-9)

        if peak_rate_hz > min_peak_rate_hz:
            total_steps += len(peaks)
            kept_windows.append((start, end, len(peaks), peak_rate_hz, seg_std))

    # global cleanup for tiny false positives
    if total_steps < min_final_steps_to_keep:
        total_steps = 0

    return {
        "steps_hat": int(total_steps),
        "filtered_signal": filt,
        "fs": fs,
        "kept_windows": kept_windows,
    }

In [5]:
rows = []

for name in gt_files:
    rec = Recording(str(train_dir / name))
    gt = int(rec.labels["step_count"])
    out = estimate_steps_windowed(rec)

    pred = out["steps_hat"]
    ape = abs(pred - gt) / max(gt, 1)

    watch_loc_id = rec.labels.get("watch_loc", None)

    activity_ids = rec.labels.get("activities", [])
    activity_names = [
        activity.name.lower()
        for activity in Activity
        if activity.value in activity_ids
    ]

    rows.append({
        "activities": activity_names,
        "watch_location": watch_loc_id,
        "gt_steps": gt,
        "pred_steps": pred,
        "abs_err": abs(pred - gt),
        "ape": ape,
    })

peak_df = pd.DataFrame(rows).sort_values("gt_steps")
print(peak_df)

# --- summary metrics ---
mae = peak_df["abs_err"].mean()                # average absolute deviation
mape = peak_df.loc[peak_df["gt_steps"] > 0, "ape"].mean()

print("\nSummary:")
print(f"Average absolute deviation (MAE): {mae:.2f}")
print(f"Average relative deviation (MAPE): {mape:.3f}")

                      activities  watch_location  gt_steps  pred_steps  \
3                             []               2         0           0   
2                      [cycling]               0         0           0   
6                      [cycling]               0         0           0   
13                     [cycling]               2         0           0   
17                     [cycling]               2         0           0   
30            [walking, cycling]               0       147         152   
16            [walking, cycling]               1       154         159   
7             [walking, cycling]               0       155         155   
18            [walking, cycling]               2       168         224   
10            [walking, cycling]               2       168         232   
8             [walking, cycling]               0       172          90   
9                             []               1       192         255   
20            [walking, cycling]      

Note: the above logic scores 0.23178 on Kaggle public leaderboard

# ML-based regression approach #

Gradient-boosted regressor that predicts the number of steps directly for each 10-second window. The total step count is the sum of per-window predictions.

### 5.1 Window-level feature extraction

Extracts time-domain and frequency-domain features for each 10-second window across all labeled recordings. Features include signal energy, peak statistics, dominant frequency, band power, and spectral entropy.

In [14]:
from scipy.signal import butter, filtfilt, find_peaks, welch
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import classification_report
from sklearn.model_selection import GroupKFold
import numpy as np
import pandas as pd

def extract_window_features(
    rec,
    file_name,
    gt_steps,
    window_s=10.0,
    low_hz=0.7,
    high_hz=3.0,
    peak_prominence=0.14,
    max_step_hz=3.0,
):
    ax = rec.data["ax"].values.astype(float)
    ay = rec.data["ay"].values.astype(float)
    az = rec.data["az"].values.astype(float)
    fs = float(rec.data["ax"].samplerate)

    mag = np.sqrt(ax**2 + ay**2 + az**2)
    mag_centered = mag - np.mean(mag)

    nyq = fs / 2.0
    b, a = butter(3, [low_hz / nyq, high_hz / nyq], btype="band")
    filt = filtfilt(b, a, mag_centered)

    win_len = int(window_s * fs)
    min_distance = int(fs / max_step_hz)

    rows = []

    for win_idx, start in enumerate(range(0, len(filt), win_len)):
        end = min(start + win_len, len(filt))
        segment = filt[start:end]

        if len(segment) < max(10, min_distance):
            continue

        peaks, props = find_peaks(
            segment,
            distance=max(1, min_distance),
            prominence=peak_prominence,
        )

        duration = len(segment) / fs
        peak_rate_hz = len(peaks) / max(duration, 1e-9)

        mean_val = float(np.mean(segment))
        std_val = float(np.std(segment))
        rms_val = float(np.sqrt(np.mean(segment**2)))
        range_val = float(np.max(segment) - np.min(segment))
        q25 = float(np.quantile(segment, 0.25))
        q75 = float(np.quantile(segment, 0.75))

        prom = props.get("prominences", np.array([]))
        prom_mean = float(np.mean(prom)) if len(prom) > 0 else 0.0
        prom_std = float(np.std(prom)) if len(prom) > 0 else 0.0
        prom_max = float(np.max(prom)) if len(prom) > 0 else 0.0

        freqs, power = welch(segment, fs=fs, nperseg=min(256, len(segment)))
        valid = (freqs >= low_hz) & (freqs <= high_hz)
        if np.any(valid):
            valid_freqs = freqs[valid]
            valid_power = power[valid]
            dom_freq = float(valid_freqs[np.argmax(valid_power)])
            band_power = float(np.sum(valid_power))
            power_share = valid_power / max(np.sum(valid_power), 1e-12)
            spec_entropy = float(-(power_share * np.log(power_share + 1e-12)).sum())
        else:
            dom_freq = 0.0
            band_power = 0.0
            spec_entropy = 0.0

        rows.append({
            "file": file_name,
            "window_idx": win_idx,
            "start_s": start / fs,
            "end_s": end / fs,
            "gt_steps_total": gt_steps,
            "peak_count": int(len(peaks)),
            "peak_rate_hz": peak_rate_hz,
            "mean": mean_val,
            "std": std_val,
            "rms": rms_val,
            "range": range_val,
            "q25": q25,
            "q75": q75,
            "prom_mean": prom_mean,
            "prom_std": prom_std,
            "prom_max": prom_max,
            "dom_freq": dom_freq,
            "band_power": band_power,
            "spec_entropy": spec_entropy,
        })

    return rows

### 5.2 Construct weakly-labeled window dataset

Assigns a regression target to each window based on the recording-level ground truth:
- **Zero-step recordings** → target = 0 for all windows
- **High-confidence recordings (≥300 steps)** → target = gt_steps distributed uniformly across windows by duration
- **Mid-range recordings (<300 steps)** → target = peak_count for windows with plausible cadence and sufficient energy, else 0

In [15]:
window_rows = []

for name in gt_files:
    rec = Recording(str(train_dir / name))
    gt = int(rec.labels["step_count"])

    rows = extract_window_features(rec, name, gt_steps=gt)

    # weak labels
    if gt == 0:
        label = 0
    elif gt >= 300:
        label = 1
    else:
        label = None  # will be filled in per-window below

    for r in rows:
        if label is not None:
            r["label"] = label
        else:
            # mid-range recording: use window signal quality as proxy
            r["label"] = 1 if (r["peak_rate_hz"] >= 1.5 and r["std"] >= 0.12) else 0
        window_rows.append(r)

window_df = pd.DataFrame(window_rows)
train_windows_df = window_df.copy()
train_windows_df["label"] = train_windows_df["label"].astype(int)

print("All windows:", len(window_df))
print("Weakly labeled windows:", len(train_windows_df))
print(train_windows_df["label"].value_counts())
train_windows_df.head()

All windows: 1389
Weakly labeled windows: 1389
label
1    793
0    596
Name: count, dtype: int64


,file,window_idx,start_s,end_s,gt_steps_total,peak_count,peak_rate_hz,mean,std,rms,range,q25,q75,prom_mean,prom_std,prom_max,dom_freq,band_power,spec_entropy,label
0,train_trace_004.pkl,0,0.000000,9.996269,1023,12,1.200448,0.001441,0.222157,0.222162,1.060415,-0.092449,0.112453,0.732614,0.248593,1.026999,1.563083,0.056144,0.801146,1
1,train_trace_004.pkl,1,9.996269,19.992537,1023,19,1.900709,-0.004767,0.318710,0.318746,1.022067,-0.318295,0.309789,0.896992,0.060971,1.003593,1.563083,0.126092,0.756095,1
2,train_trace_004.pkl,2,19.992537,29.988806,1023,20,2.000747,0.006513,0.319045,0.319112,1.039400,-0.306774,0.323016,0.864316,0.117853,0.996525,2.344625,0.124135,0.748361,1
3,train_trace_004.pkl,3,29.988806,39.985075,1023,19,1.900709,-0.001451,0.341410,0.341413,1.066712,-0.340737,0.340040,0.956791,0.051734,1.042375,2.344625,0.142364,0.739188,1
4,train_trace_004.pkl,4,39.985075,49.981343,1023,19,1.900709,-0.004856,0.356548,0.356581,1.134648,-0.356987,0.348002,1.002928,0.061799,1.102373,1.563083,0.156652,0.747299,1


### 5.3 Train and cross-validate the regressor

Fits a GradientBoostingRegressor to predict per-window step counts. GroupKFold cross-validation is used so that windows from the same recording never appear in both train and validation splits — this gives an honest estimate of generalisation.

In [16]:
feature_cols = [
    "peak_count",
    "peak_rate_hz",
    "mean",
    "std",
    "rms",
    "range",
    "q25",
    "q75",
    "prom_mean",
    "prom_std",
    "prom_max",
    "dom_freq",
    "band_power",
    "spec_entropy",
]

X = train_windows_df[feature_cols].fillna(0.0)
y = train_windows_df["label"].values
groups = train_windows_df["file"].values

reg_rows = []
for name in gt_files:
    rec = Recording(str(train_dir / name))
    gt = int(rec.labels["step_count"])
    rows = extract_window_features(rec, name, gt_steps=gt)
    if not rows:
        continue

    total_duration = sum(r["end_s"] - r["start_s"] for r in rows)

    for r in rows:
        win_duration = r["end_s"] - r["start_s"]

        if gt == 0:
            r["target_steps"] = 0.0
        elif gt >= 300:
            # distribute gt proportionally across windows
            r["target_steps"] = gt * (win_duration / total_duration)
        else:
            # mid-range: only include windows with plausible cadence
            if r["peak_rate_hz"] >= 1.5 and r["std"] >= 0.12:
                r["target_steps"] = r["peak_count"]  # trust the peaks
            else:
                r["target_steps"] = 0.0

        reg_rows.append(r)

reg_df = pd.DataFrame(reg_rows)

X_reg = reg_df[feature_cols].fillna(0.0)
y_reg = reg_df["target_steps"].values
groups_reg = reg_df["file"].values

reg = GradientBoostingRegressor(
    n_estimators=200,
    max_depth=4,
    min_samples_leaf=3,
    random_state=42,
)

# cross-val to check it's not overfit
cv = GroupKFold(n_splits=5)
fold_maes = []
for fold, (tr_idx, va_idx) in enumerate(cv.split(X_reg, y_reg, groups_reg), 1):
    reg.fit(X_reg.iloc[tr_idx], y_reg[tr_idx])
    preds = reg.predict(X_reg.iloc[va_idx])
    mae = np.mean(np.abs(preds - y_reg[va_idx]))
    fold_maes.append(mae)
    print(f"Fold {fold}: window-level MAE = {mae:.2f}")

print(f"\nMean CV window MAE: {np.mean(fold_maes):.2f} ± {np.std(fold_maes):.2f}")

reg.fit(X_reg, y_reg)  # final fit on all data

Fold 1: window-level MAE = 3.96
Fold 2: window-level MAE = 1.81
Fold 3: window-level MAE = 2.41
Fold 4: window-level MAE = 1.95
Fold 5: window-level MAE = 3.23

Mean CV window MAE: 2.67 ± 0.81


,loss,'squared_error'
,learning_rate,0.1
,n_estimators,200
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,3
,min_weight_fraction_leaf,0.0
,max_depth,4
,min_impurity_decrease,0.0
,init,None


### 5.4 Inference

Applies the trained regressor to a new recording. Each window gets an independent step-count prediction; the final estimate is the sum across all windows. A minimum threshold suppresses false positives on zero-step recordings.

In [17]:
def estimate_steps_regression(
    rec,
    reg,
    feature_cols,
    window_s=10.0,
    low_hz=0.7,
    high_hz=3.0,
    peak_prominence=0.14,
    max_step_hz=3.0,
    min_final_steps_to_keep=80,
):
    rows = extract_window_features(
        rec=rec,
        file_name="current_recording",
        gt_steps=-1,
        window_s=window_s,
        low_hz=low_hz,
        high_hz=high_hz,
        peak_prominence=peak_prominence,
        max_step_hz=max_step_hz,
    )

    if len(rows) == 0:
        return {"steps_hat": 0, "window_df": pd.DataFrame()}

    wdf = pd.DataFrame(rows)
    Xw = wdf[feature_cols].fillna(0.0)

    # regressor predicts steps per window directly
    predicted_steps_per_window = reg.predict(Xw)
    predicted_steps_per_window = np.maximum(predicted_steps_per_window, 0)  # no negatives

    total_steps = int(np.round(predicted_steps_per_window.sum()))

    if total_steps < min_final_steps_to_keep:
        total_steps = 0

    return {"steps_hat": total_steps, "window_df": wdf}

## 6. Compare baseline vs regression estimator

Side-by-side comparison of the peak-detection baseline and the regression approach across all 33 labeled recordings. Reports per-recording absolute error and APE, plus aggregate MAE and MAPE for both methods.

In [18]:
# --- compare baseline vs hybrid ---
comparison_rows = []

for name in gt_files:
    rec = Recording(str(train_dir / name))
    gt = int(rec.labels["step_count"])

    baseline_out = estimate_steps_windowed(rec)
    hybrid_out = estimate_steps_regression(rec, reg=reg, feature_cols=feature_cols)

    baseline_pred = int(baseline_out["steps_hat"])
    hybrid_pred = int(hybrid_out["steps_hat"])

    comparison_rows.append({
        "file": name,
        "gt_steps": gt,
        "baseline_pred": baseline_pred,
        "hybrid_pred": hybrid_pred,
        "baseline_abs_err": abs(baseline_pred - gt),
        "hybrid_abs_err": abs(hybrid_pred - gt),
        "baseline_ape": abs(baseline_pred - gt) / max(gt, 1),
        "hybrid_ape": abs(hybrid_pred - gt) / max(gt, 1),
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values("gt_steps")
print(comparison_df)

baseline_mae = comparison_df["baseline_abs_err"].mean()
hybrid_mae = comparison_df["hybrid_abs_err"].mean()

baseline_mape = comparison_df.loc[comparison_df["gt_steps"] > 0, "baseline_ape"].mean()
hybrid_mape = comparison_df.loc[comparison_df["gt_steps"] > 0, "hybrid_ape"].mean()

print("\nSummary:")
print(f"Baseline MAE : {baseline_mae:.2f}")
print(f"Hybrid MAE   : {hybrid_mae:.2f}")
print(f"Baseline MAPE: {baseline_mape:.3f}")
print(f"Hybrid MAPE  : {hybrid_mape:.3f}")

                   file  gt_steps  baseline_pred  hybrid_pred  \
2   train_trace_017.pkl         0              0            0   
3   train_trace_021.pkl         0              0            0   
6   train_trace_086.pkl         0              0            0   
17  train_trace_185.pkl         0              0            0   
13  train_trace_131.pkl         0              0            0   
30  train_trace_361.pkl       147            152          117   
16  train_trace_161.pkl       154            159          135   
7   train_trace_090.pkl       155            155           80   
10  train_trace_116.pkl       168            232          240   
18  train_trace_200.pkl       168            224          213   
8   train_trace_093.pkl       172             90           84   
9   train_trace_110.pkl       192            255          251   
20  train_trace_212.pkl       245            268          246   
29  train_trace_354.pkl       255            250          247   
32  train_trace_388.pkl  

submission

In [6]:
# --- step-count-only baseline submission ---
import re
from pathlib import Path
import pandas as pd

test_dir = Path("data/test")
submission_name = "submission_step_count_ML1.csv"

def parse_trace_id(path: Path) -> int:
    match = re.search(r"(\d+)\.pkl$", path.name)
    if match is None:
        raise ValueError(f"Could not parse Id from filename: {path.name}")
    return int(match.group(1))

rows = []

for path in sorted(test_dir.glob("*.pkl")):
    rec = Recording(str(path))
    step_pred = int(estimate_steps_windowed(rec)["steps_hat"]) #(only submits baseline)
    # step_pred = int(estimate_steps_regression(
    #     rec,
    #     reg=reg,
    #     feature_cols=feature_cols
    # )["steps_hat"])

    rows.append({
        "Id": parse_trace_id(path),
        "watch_loc": -1,      # default placeholder
        "path_idx": -1,       # default placeholder
        "standing": -1,   # default placeholder
        "walking": -1,    # default placeholder
        "running": -1,    # default placeholder
        "cycling": -1,    # default placeholder
        "step_count": step_pred,
    })

submission_df = pd.DataFrame(rows).sort_values("Id")

# optional sanity checks
print(submission_df.head())
print(f"\nRows: {len(submission_df)}")
print(f"Step count min/max: {submission_df['step_count'].min()} / {submission_df['step_count'].max()}")

submission_df.to_csv(submission_name, index=False)
print(f"\nSaved submission to {submission_name}")

   Id  watch_loc  path_idx  standing  walking  running  cycling  step_count
0   0         -1        -1        -1       -1       -1       -1         975
1   1         -1        -1        -1       -1       -1       -1         253
2   2         -1        -1        -1       -1       -1       -1         376
3   3         -1        -1        -1       -1       -1       -1        1093
4   4         -1        -1        -1       -1       -1       -1        1012

Rows: 280
Step count min/max: 0 / 1430

Saved submission to submission_step_count_ML1.csv
